# 第 1 章 实践基础

深度学习在很多领域都有非常出色的表现，在图像识别、语音识别、自然语言处理、机器人、广告投放、医学诊断和金融等领域都有广泛应用。而目前深度学习的模型还主要是各种各样的神经网络。随着网络越来越复杂，从底层开始一步步实现深度学习系统变得非常低效，其中涉及模型搭建、梯度求解、并行计算、代码实现等多个环节。每一个环节都需要进行精心实现和检查，耗费了开发人员很多精力。为此，**深度学习框架**（也常称为**机器学习框架**）应运而生。它的优势主要体现在两方面：

- **实现简单**：深度学习框架屏蔽了底层实现，用户只需关注模型的逻辑结构，同时简化了计算逻辑，降低了深度学习入门门槛。
- **使用高效**：深度学习框架具备灵活的移植性，可以在不同设备（CPU、GPU 或移动端）之间无缝迁移，使得模型训练和部署更高效。

本书使用 **PyTorch** 作为实践的基础框架。PyTorch 由 Meta 于 2016 年开源，是目前学术界和工业界使用最广泛的深度学习框架之一，凭借动态计算图、简洁的 Python API 和庞大的生态系统，已经成为深度学习研究和应用的事实标准。

本章先对实践环节的基础知识进行介绍，主要包括以下内容：

- **张量**（Tensor）：一种多维数组，是深度学习中表示和存储数据的主要形式。在动手实践机器学习之前，需要先熟悉张量的概念、性质和运算规则，了解 PyTorch 中张量的各种 API。
- **算子**（Operator，Op）：构建神经网络模型的基础组件。每个算子有前向和反向计算过程，前向计算对应一个数学函数，反向计算对应这个数学函数的梯度计算。有了算子，我们就可以很方便地通过它们搭建复杂的神经网络模型，而不需要手工计算梯度。

此外，本章还会介绍本书自定义的数据集容器、轻量级训练框架 `Runner` 类以及模型保存与加载的方法。


## 1.1 如何运行本书的代码

本书涉及大量代码实践，通过运行代码理解如何构建模型及训练网络。推荐使用 Jupyter Notebook 来运行本书的代码。

### 1.1.1 环境准备

本书代码基于 Python 与 PyTorch 框架。请先确认运行环境满足以下条件：

- 操作系统：Linux、macOS 或 Windows
- Python 版本：3.11 及以上（64-bit）
- pip 版本：25 及以上
- PyTorch 版本：2.7 及以上
- torchvision 版本：与 PyTorch 对应的版本
- GPU 运行时（可选）：CUDA 12.6 及以上，并安装与之匹配的 NVIDIA 驱动

本书后续代码示例和 API 说明均以 PyTorch 2.7 及以上版本为基准。若早期教程或旧版代码中出现已经过时的参数、类名或调用方式，应优先采用当前 PyTorch 与 torchvision 官方推荐的写法。

第 1 章的内容使用 CPU 即可完成。从第 2 章开始的训练任务建议使用支持 CUDA 的 GPU；如果本地没有 GPU，可以使用 Google Colab、Kaggle 等在线平台提供的免费 GPU 资源。

通过如下命令安装 CPU 版本的 PyTorch：

```bash
pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
```

如果有 GPU，请参考 PyTorch 官网 <https://pytorch.org/get-started/locally/> 选择匹配自己 CUDA 版本的安装命令。

安装完成后，可以在 Python 解释器中输入以下命令验证：

```python
import torch
print(torch.__version__)
print(torch.cuda.is_available())   # 是否检测到 GPU
```

### 1.1.2 代码下载与使用方法

本书所有代码都开源在 GitHub：<https://github.com/nndl/nndl-practice>。建议先 clone 仓库到本地，再用 Jupyter Notebook 打开各章的 `.ipynb` 文件。

```bash
git clone https://github.com/nndl/nndl-practice.git
cd nndl-practice/pytorch
pip install -r requirements.txt
jupyter notebook
```

有关本书的问题和讨论，欢迎在项目的 Issue 中提出。


## 1.2 张量

在深度学习的实践中，通常使用向量或矩阵运算来提高计算效率。比如 $w_1 x_1 + w_2 x_2 + \cdots + w_D x_D$ 的计算可以用 $\boldsymbol{w}^\top \boldsymbol{x}$ 来代替（其中 $\boldsymbol{w}=[w_1, w_2, \cdots, w_D]^\top, \boldsymbol{x}=[x_1, x_2, \cdots, x_D]^\top$），这样可以充分利用计算机的并行计算能力，特别是利用 GPU 来实现高效矩阵运算。

在深度学习框架中，数据经常用**张量**（Tensor）的形式来存储。张量是矩阵的扩展与延伸，可以看作是高阶的矩阵：1 阶张量为向量，2 阶张量为矩阵。如果你对 NumPy 熟悉，那么张量就类似于 NumPy 中的**多维数组**（ndarray），可以具有任意多的维度。

> **注意**：这里的"维度"是"阶"的概念，和线性代数中向量的"维度"含义不同。

张量的大小可以用**形状**（shape）来描述。比如一个三维张量的形状是 `[2, 2, 5]`，表示每一维（也称为**轴**，axis）的元素的数量，即第 0 轴上元素数量是 2，第 1 轴上元素数量是 2，第 2 轴上元素数量是 5。

张量中元素的类型可以是布尔型数据、整数、浮点数或者复数，但**同一张量中所有元素的数据类型均相同**。因此我们可以给张量定义一个**数据类型**（dtype）来表示其元素的类型。


### 1.2.1 创建张量

创建张量可以有多种方式，如指定数据创建、指定形状创建、指定区间创建等。

#### 1.2.1.1 指定数据创建张量

通过给定的 Python 列表数据，可以创建任意维度的张量。

**1）一维张量**：通过指定的 Python 列表数据 `[2.0, 3.0, 4.0]`，创建一个一维张量。


In [ ]:
# 导入 PyTorch
import torch

# 创建一维张量
ndim_1_Tensor = torch.tensor([2.0, 3.0, 4.0])
print(ndim_1_Tensor)


**2）二维张量**：通过指定的 Python 列表数据来创建类似矩阵（matrix）的二维张量。


In [ ]:
# 创建二维张量
ndim_2_Tensor = torch.tensor([[1.0, 2.0, 3.0],
                              [4.0, 5.0, 6.0]])
print(ndim_2_Tensor)


**3）多维张量**：同样可以创建维度为 3、4、…、$N$ 等更复杂的多维张量。


In [ ]:
# 创建三维张量
ndim_3_Tensor = torch.tensor([[[1, 2, 3, 4, 5],
                               [6, 7, 8, 9, 10]],
                              [[11, 12, 13, 14, 15],
                               [16, 17, 18, 19, 20]]])
print(ndim_3_Tensor)
print('shape:', ndim_3_Tensor.shape)


需要注意的是，张量在任何一个维度上的元素数量必须相等。下面尝试定义一个在同一维度上元素数量不等的张量，会抛出异常。


In [ ]:
# 尝试定义在不同维度上元素数量不等的张量（会报错）
try:
    bad = torch.tensor([[1.0, 2.0],
                        [4.0, 5.0, 6.0]])
except (ValueError, TypeError) as e:
    print('捕获到异常：', e)


#### 1.2.1.2 指定形状创建张量

如果要创建一个指定形状、元素数据相同的张量，可以使用 `torch.zeros`、`torch.ones`、`torch.full` 等 API。


In [ ]:
m, n = 2, 3

# torch.zeros：全 0，形状为 (m, n)
zeros_Tensor = torch.zeros(m, n)

# torch.ones：全 1
ones_Tensor = torch.ones(m, n)

# torch.full：全部填充为指定值
full_Tensor = torch.full((m, n), 10)

print('zeros Tensor:\n', zeros_Tensor)
print('ones  Tensor:\n', ones_Tensor)
print('full  Tensor:\n', full_Tensor)


#### 1.2.1.3 指定区间创建张量

如果要在指定区间内创建张量，可以使用 `torch.arange`、`torch.linspace` 等 API。


In [ ]:
# torch.arange：以步长 step 均匀分隔区间 [start, end)，得到一维张量
arange_Tensor = torch.arange(start=1, end=5, step=1)

# torch.linspace：将区间 [start, end] 等分为 steps 份，得到一维张量
linspace_Tensor = torch.linspace(start=1, end=5, steps=5)

print('arange   Tensor:', arange_Tensor)
print('linspace Tensor:', linspace_Tensor)


### 1.2.2 张量的属性

#### 1.2.2.1 张量的形状

张量具有如下形状属性：

- `Tensor.ndim`：张量的维度（阶），例如向量的维度为 1，矩阵的维度为 2。
- `Tensor.shape`：张量每个维度上元素的数量。
- `Tensor.shape[n]`：张量第 $n$ 维的大小。
- `Tensor.numel()`：张量中全部元素的个数。

下面创建一个四维张量，并打印它的几个形状属性。


In [ ]:
ndim_4_Tensor = torch.ones(2, 3, 4, 5)

print('Number of dimensions:', ndim_4_Tensor.ndim)
print('Shape of Tensor     :', ndim_4_Tensor.shape)
print('Elements along axis 0:', ndim_4_Tensor.shape[0])
print('Elements along last axis:', ndim_4_Tensor.shape[-1])
print('Number of elements  :', ndim_4_Tensor.numel())


#### 1.2.2.2 形状的改变

除了查看张量的形状外，重新设置张量的形状也具有重要意义。可以调用 `torch.reshape` 函数（或 `Tensor.reshape` / `Tensor.view` 方法）改变张量的形状。


In [ ]:
# 定义一个形状为 [3, 2, 5] 的三维张量
ndim_3_Tensor = torch.tensor([[[1, 2, 3, 4, 5],
                               [6, 7, 8, 9, 10]],
                              [[11, 12, 13, 14, 15],
                               [16, 17, 18, 19, 20]],
                              [[21, 22, 23, 24, 25],
                               [26, 27, 28, 29, 30]]])
print('the shape of ndim_3_Tensor:', ndim_3_Tensor.shape)

# 在保持数据不变的情况下改变形状
reshape_Tensor = torch.reshape(ndim_3_Tensor, (2, 5, 3))
print('After reshape:\n', reshape_Tensor)


从输出结果看，将张量从 `[3, 2, 5]` reshape 成 `[2, 5, 3]` 时，张量内的数据本身没有改变，元素顺序也没有改变，只有数据形状发生了改变。

> **小技巧**：使用 reshape 时，可以把某一个维度填成 `-1`，表示这个维度的值由张量的元素总数和剩余维度自动推断出来。**有且只有一个维度可以被设置为 -1**。


In [ ]:
# -1 表示自动推断
new_Tensor1 = ndim_3_Tensor.reshape(-1)
print('new Tensor 1 shape:', new_Tensor1.shape)

# 也可以直接显式给出每个维度的大小
new_Tensor2 = ndim_3_Tensor.reshape(3, 5, 2)
print('new Tensor 2 shape:', new_Tensor2.shape)


除使用 `reshape` 外，还可以通过 `torch.unsqueeze` 在张量的某个维度上插入尺寸为 1 的新维度，以及通过 `torch.squeeze` 把所有尺寸为 1 的维度去掉。


In [ ]:
ones_Tensor = torch.ones(5, 10)

# 在第 0 维插入一个尺寸为 1 的新维度
new_Tensor1 = torch.unsqueeze(ones_Tensor, dim=0)
print('new Tensor 1 shape:', new_Tensor1.shape)

# 也可以连续插入多次
new_Tensor2 = ones_Tensor.unsqueeze(1).unsqueeze(2)
print('new Tensor 2 shape:', new_Tensor2.shape)

# squeeze 去掉所有为 1 的维
print('squeeze        :', new_Tensor2.squeeze().shape)


#### 1.2.2.3 张量的数据类型

PyTorch 通过 `Tensor.dtype` 查看张量的数据类型，支持 `bool`、`float16`、`float32`、`float64`、`uint8`、`int8`、`int16`、`int32`、`int64` 和复数类型等。

- 通过 Python 元素创建的张量，可以用 `dtype` 参数指定数据类型；如果未指定：
  - 对于 Python 整数，默认会创建 `int64` 型张量。
  - 对于 Python 浮点数，默认会创建 `float32` 型张量。
- 通过 NumPy 数组创建的张量，与其原来的数据类型相同。使用 `torch.from_numpy` 可以把 NumPy 数组转化为张量。


In [ ]:
# 通过已知数据来创建张量并查看 dtype
print('Tensor dtype from Python integers      :', torch.tensor(1).dtype)
print('Tensor dtype from Python floating point:', torch.tensor(1.0).dtype)


如果想改变张量的数据类型，可以调用 `Tensor.to(dtype)` 方法。


In [ ]:
# 定义 dtype 为 float32 的张量
float32_Tensor = torch.tensor(1.0)

# 转换为 int64
int64_Tensor = float32_Tensor.to(torch.int64)
print('Tensor after cast to int64:', int64_Tensor.dtype)


#### 1.2.2.4 张量的设备位置

初始化张量时可以通过 `device` 参数指定其分配的设备位置，可支持的设备主要有两类：**CPU** 和 **GPU**（CUDA 设备）。

未指定设备位置时，张量默认创建在 CPU 上。要把张量移到 GPU，使用 `tensor.to('cuda')` 或 `tensor.cuda()`。设备位置不一致的张量之间不能直接做运算。


In [ ]:
# 创建 CPU 上的张量
cpu_Tensor = torch.tensor(1, device='cpu')
print('cpu Tensor:', cpu_Tensor.device)

# 如果检测到 GPU，再演示一下
if torch.cuda.is_available():
    gpu_Tensor = torch.tensor(1, device='cuda')
    print('gpu Tensor:', gpu_Tensor.device)
else:
    print('当前环境未检测到 GPU，跳过 GPU 张量创建演示。')


### 1.2.3 张量与 NumPy 数组转换

张量和 NumPy 数组可以相互转换：

- `torch.from_numpy(ndarray)` 把 NumPy 数组转化为张量。**注意**：得到的张量与原 NumPy 数组共享底层内存，对其中一个的修改会影响另一个。
- `Tensor.numpy()` 把张量转化为 NumPy 数组（仅限 CPU 张量；如果张量在 GPU 上需要先 `.cpu()`）。


In [ ]:
import numpy as np

ndim_1_Tensor = torch.tensor([1., 2.])

# Tensor -> NumPy
print('Tensor to NumPy:', ndim_1_Tensor.numpy())

# NumPy -> Tensor
arr = np.array([3., 4., 5.])
print('NumPy to Tensor:', torch.from_numpy(arr))


### 1.2.4 张量的访问

#### 1.2.4.1 索引和切片

可以通过索引或切片操作方便地访问、修改张量。PyTorch 使用标准的 Python 索引规则和 NumPy 索引规则，具有以下特点：

1. 基于 `0 ~ n-1` 的下标进行索引，下标为负数时从尾部开始计算。
2. 通过冒号 `:` 分隔切片参数 `start:stop:step` 来进行切片操作，三个参数均可缺省。

针对一维张量，对单个轴进行索引和切片。


In [ ]:
# 定义一维张量
ndim_1_Tensor = torch.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8])

print('Origin Tensor:', ndim_1_Tensor)
print('First element:', ndim_1_Tensor[0])
print('Last element :', ndim_1_Tensor[-1])
print('All element  :', ndim_1_Tensor[:])
print('Before 3     :', ndim_1_Tensor[:3])
print('Interval of 3:', ndim_1_Tensor[::3])
print('Reverse      :', ndim_1_Tensor.flip(0))   # PyTorch 不支持 [::-1]，用 flip 等价


> **小细节**：与 NumPy 不同，PyTorch 暂时不支持用负步长 `[::-1]` 来反转张量，需要用 `torch.flip` 或 `Tensor.flip(dim)`。

针对二维及以上维度的张量，可以在多个维度上进行索引或切片。索引或切片的第一个值对应第 0 维，第二个值对应第 1 维，依此类推；如果某个维度上未指定索引，则默认为 `:`。


In [ ]:
# 定义二维张量
ndim_2_Tensor = torch.tensor([[0, 1, 2, 3],
                              [4, 5, 6, 7],
                              [8, 9, 10, 11]])

print('Origin Tensor:\n', ndim_2_Tensor)
print('First row     :', ndim_2_Tensor[0])
print('First row     :', ndim_2_Tensor[0, :])
print('First column  :', ndim_2_Tensor[:, 0])
print('Last column   :', ndim_2_Tensor[:, -1])
print('Element [0,1] :', ndim_2_Tensor[0, 1])


#### 1.2.4.2 修改张量

与访问张量类似，可以在单个或多个轴上通过索引或切片操作来修改张量。

> **注意**：通过索引或切片操作修改张量是**原地（in-place）操作**，原值不会被保存。如果被修改的张量参与梯度计算，将仅使用修改后的数值，这可能会给梯度计算引入风险，使用时要小心。


In [ ]:
ndim_2_Tensor = torch.ones(2, 3, dtype=torch.float32)
print('Origin Tensor:\n', ndim_2_Tensor)

# 把第 0 行整体改成 0
ndim_2_Tensor[0] = 0
print('change Tensor:\n', ndim_2_Tensor)

# 用切片改：把第 0 行改成 2.1
ndim_2_Tensor[0:1] = 2.1
print('change Tensor:\n', ndim_2_Tensor)

# 全部改成 3
ndim_2_Tensor[...] = 3
print('change Tensor:\n', ndim_2_Tensor)


### 1.2.5 张量的运算

张量支持基础数学运算、逻辑运算、矩阵运算等百余种运算。以加法为例，有两种常用调用方式：

1. 使用 PyTorch 函数 API：`torch.add(x, y)`。
2. 使用张量类成员函数：`x.add(y)`，或更常用的算术运算符 `x + y`。


In [ ]:
# 定义两个张量
x = torch.tensor([[1.1, 2.2], [3.3, 4.4]], dtype=torch.float64)
y = torch.tensor([[5.5, 6.6], [7.7, 8.8]], dtype=torch.float64)

# 三种等价写法
print('Method 1 (torch.add):\n', torch.add(x, y))
print('Method 2 (x.add)    :\n', x.add(y))
print('Method 3 (x + y)    :\n', x + y)


从输出结果看，三种调用方式效果相同。下面以张量类成员函数和运算符的角度来介绍常用张量操作。

#### 1.2.5.1 数学运算

张量类的基础数学函数如下：

```python
x.abs()           # 逐元素取绝对值
x.ceil()          # 逐元素向上取整
x.floor()         # 逐元素向下取整
x.round()         # 逐元素四舍五入
x.exp()           # 逐元素计算自然指数
x.log()           # 逐元素计算自然对数
x.reciprocal()    # 逐元素求倒数
x.square()        # 逐元素计算平方
x.sqrt()          # 逐元素计算平方根
x.sin()           # 逐元素计算正弦
x.cos()           # 逐元素计算余弦
x.add(y)          # 逐元素加
x.sub(y)          # 逐元素减
x.mul(y)          # 逐元素乘
x.div(y)          # 逐元素除
x.fmod(y)         # 逐元素除并取余
x.pow(y)          # 逐元素幂
x.max()           # 指定维度上元素最大值，默认为全部维度
x.min()           # 指定维度上元素最小值，默认为全部维度
x.prod()          # 指定维度上元素累乘
x.sum()           # 指定维度上元素的和
```

PyTorch 同样对 Python 数学运算相关的魔法函数进行了重写，下面写法与上面等价：

```python
x + y    -> x.add(y)
x - y    -> x.sub(y)
x * y    -> x.mul(y)
x / y    -> x.div(y)
x % y    -> x.fmod(y)
x ** y   -> x.pow(y)
```


#### 1.2.5.2 逻辑运算

张量类的逻辑运算函数如下：

```python
torch.isfinite(x)   # 判断元素是否是有限的数字（不是 inf / nan）
torch.equal(x, y)   # 两个张量的全部元素是否都相等，返回 Python bool
x.eq(y)             # 逐元素是否相等，返回相同形状的布尔型张量
x.ne(y)             # 逐元素是否不相等
x.lt(y)             # 逐元素是否小于
x.le(y)             # 逐元素是否小于等于
x.gt(y)             # 逐元素是否大于
x.ge(y)             # 逐元素是否大于等于
torch.allclose(x, y)  # 是否在容忍误差内全部接近，返回 Python bool
```

Python 比较运算符 `==`、`!=`、`<`、`<=`、`>`、`>=` 也都被重载，使用上更直观。


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([1.0, 2.5, 3.0])

print('x == y :', x == y)
print('x <= y :', x <= y)
print('allclose:', torch.allclose(x, y))


#### 1.2.5.3 矩阵运算

张量类还包含矩阵运算相关的函数，如矩阵的转置、范数计算和乘法等。

```python
x.t()                       # 二维矩阵转置
x.transpose(0, 1)           # 交换第 0 维与第 1 维
torch.norm(x, p='fro')      # 矩阵的弗罗贝尼乌斯范数
torch.dist(x, y, p=2)       # 矩阵 (x-y) 的 2-范数
x.matmul(y)                 # 矩阵乘积，等价于 x @ y
```

有些矩阵运算也支持大于两维的张量，比如 `matmul` 函数会对最后两个维度做矩阵乘，前面的维度按广播规则处理。比如 `x` 是形状为 $[j, k, n, m]$ 的张量，`y` 是形状为 $[j, k, m, p]$ 的张量，则 `x.matmul(y)` 输出的张量形状为 $[j, k, n, p]$。


In [ ]:
A = torch.randn(3, 4)
B = torch.randn(4, 5)
print('A @ B shape :', (A @ B).shape)
print('A.matmul(B) :', A.matmul(B).shape)

# 转置与范数
print('A.t() shape :', A.t().shape)
print('Frobenius   :', torch.norm(A, p='fro').item())


#### 1.2.5.4 广播机制

PyTorch 的很多 API 在计算时支持**广播**（broadcasting）机制，允许在一些运算中使用不同形状的张量。直观地说，如果一个形状较小的张量与一个形状较大的张量一起做运算，框架会先把小的那一个"扩展"成与大的一致，再做逐元素运算。

**广播机制的条件**（与 NumPy 一致）：

1. 每个张量至少为一维张量。
2. 从最右维往左对齐比较两个张量的形状，对应维度的大小要么相等，要么其中一个是 1，要么其中一个不存在。

满足这些条件的张量就可以广播。


In [ ]:
# 1. 两个形状一致的张量当然可以广播
x = torch.ones(2, 3, 4)
y = torch.ones(2, 3, 4)
z = x + y
print('broadcasting with two same shape tensor:', z.shape)

# 2. 不同形状但可广播
x = torch.ones(2, 3, 1, 5)
y = torch.ones(   3, 4, 1)
# 从后往前比较：
#  第 1 次：y 是 1，可以
#  第 2 次：x 是 1，可以
#  第 3 次：都是 3，相等
#  第 4 次：y 不存在，可以
z = x + y
print('broadcasting with two different shape tensor:', z.shape)


定义两个形状分别为 `[2, 3, 4]` 和 `[2, 3, 6]` 的张量，观察它们是否能够通过广播相加。


In [ ]:
x = torch.ones(2, 3, 4)
y = torch.ones(2, 3, 6)
try:
    z = x + y
except RuntimeError as e:
    print('捕获到广播失败：', e)


从输出结果看，此时 `x` 和 `y` 不能广播，因为在最右维上 4 和 6 不相等也不为 1。

**广播机制的计算规则**：

1. 如果两个张量的维度数不一致，那么先在较短形状的前面（左侧）补 1，直到两个张量的维度数相等。
2. 在两个张量的形状长度相等之后，每个维度上的结果维度就是该维度上较大的那个。

以张量 `x` 形状 `[2, 3, 1, 5]` 和张量 `y` 形状 `[3, 4, 1]` 为例。先把 `y` 的形状补齐为 `[1, 3, 4, 1]`，再逐维比较：

- 第 0 维：max(2, 1) = 2
- 第 1 维：max(3, 3) = 3
- 第 2 维：max(1, 4) = 4
- 第 3 维：max(5, 1) = 5

结果形状为 `[2, 3, 4, 5]`。

矩阵乘积函数 `torch.matmul` 在深度学习中使用非常多，它的广播规则需要特别说明：

1. 两个张量均为一维：返回点积结果（标量）。
2. 两个张量都是二维：返回矩阵乘积。
3. `x` 是一维、`y` 是二维：先把 `x` 形状改成 `[1, D]`，做矩阵乘后再去掉前置维度。
4. `x` 是二维、`y` 是一维：返回矩阵与向量的乘积。
5. 两个张量都是 $N > 2$ 维：按广播规则广播除最后两维之外的其它维度。比如 `x` 形状是 $[j, 1, n, m]$，`y` 形状是 $[k, m, p]$，结果形状是 $[j, k, n, p]$。


In [ ]:
x = torch.ones(10, 1, 5, 2)
y = torch.ones(    3, 2, 5)
z = torch.matmul(x, y)
print('After matmul:', z.shape)


> **原地操作与非原地操作**：PyTorch 中很多 API 都有原地（in-place）操作和非原位操作之分。原地操作直接在原张量上保存结果，名称后面带一个下划线 `_`，比如 `x.add_(y)`；非原位操作则不修改原张量，比如 `x.add(y)` 或 `x + y`。

**\\练一练\\**：尝试本节中的各种张量运算，特别是掌握张量计算时的广播机制。


## 1.3 算子

一个复杂的机器学习模型（比如神经网络）可以看作一个复合函数，输入是数据特征，输出是标签的值或概率。为简单起见，假设一个由 $L$ 个函数复合的神经网络定义为

$$ y = f_L(\cdots f_2(f_1(x))), $$

其中 $f_l(\cdot)$ 可以为带参数的函数，也可以为不带参数的函数，$x$ 为输入特征，$y$ 为某种损失。我们将从 $x$ 到 $y$ 的计算看作一个**前向计算**过程，而神经网络的参数学习需要计算损失函数对所有参数的偏导数（即梯度）。

假设函数 $a_l = f_l(a_{l-1})$ 包含参数 $\theta_l$，根据链式法则

$$ \frac{\partial y}{\partial \theta_l} = \frac{\partial a_l}{\partial \theta_l} \cdot \frac{\partial y}{\partial a_l}. $$

在实践中，一种比较高效的计算 $y$ 关于 $a_l$ 的偏导数的方式是利用递归进行反向计算。令 $\delta_l \triangleq \frac{\partial y}{\partial a_l}$，则有

$$ \delta_{l-1} = \frac{\partial a_l}{\partial a_{l-1}} \delta_l \triangleq b_l(\delta_l). $$

如果将函数 $a_l = f_l(a_{l-1})$ 称为**前向函数**，则函数 $\delta_{l-1} = b_l(\delta_l)$ 称为其对应的**反向函数**。如果我们实现每个基础函数的前向函数和反向函数，就可以非常方便地通过这些基础函数组合出复杂函数，并通过链式法则反向计算复杂函数的偏导数。在深度学习框架中，这些基础函数的实现称为**算子**。有了算子，就可以像搭积木一样构建复杂的模型。


### 1.3.1 算子定义

**算子**是构建复杂机器学习模型的基础组件，它包含一个函数 $f(x)$ 的前向函数和反向函数。为了更便捷地进行算子组合，本书定义算子 `Op` 的接口如下：


In [ ]:
class Op(object):
    def __init__(self):
        pass

    def __call__(self, inputs):
        return self.forward(inputs)

    # 前向函数
    # 输入：张量 inputs
    # 输出：张量 outputs
    def forward(self, inputs):
        raise NotImplementedError

    # 反向函数
    # 输入：最终输出对 outputs 的梯度 outputs_grads
    # 输出：最终输出对 inputs 的梯度 inputs_grads
    def backward(self, outputs_grads):
        raise NotImplementedError


在上面的接口中，`forward` 是自定义 `Op` 的前向函数，必须被子类重写，参数为输入对象，参数的类型和数量任意；`backward` 是自定义 `Op` 的反向函数，必须被子类重写，参数是 `forward` 输出张量的梯度 `outputs_grads`，输出是 `forward` 输入张量的梯度 `inputs_grads`。

下面我们以 $g = \exp(a \times b + c \times d)$ 为例，分别实现加法、乘法和指数运算三个算子，再通过算子组合计算 $g$。


#### 1.3.1.1 加法算子

**前向计算**：$z = x + y$。

**反向计算**：假设经过一个其他操作后，最终输出为 $L$，令 $\delta_z = \frac{\partial L}{\partial z}$，$\delta_x = \frac{\partial L}{\partial x}$，$\delta_y = \frac{\partial L}{\partial y}$。

加法算子的反向计算的输入是梯度 $\delta_z$，输出是梯度 $\delta_x$ 和 $\delta_y$。根据链式法则：$\delta_x = \delta_z \times 1$，$\delta_y = \delta_z \times 1$。


In [ ]:
class add(Op):
    def __init__(self):
        super().__init__()

    def __call__(self, x, y):
        return self.forward(x, y)

    def forward(self, x, y):
        self.x = x
        self.y = y
        outputs = x + y
        return outputs

    def backward(self, grads):
        grads_x = grads * 1
        grads_y = grads * 1
        return grads_x, grads_y


定义 $x = 1$、$y = 4$，根据反向计算得到 $x$、$y$ 的梯度。


In [ ]:
x = 1
y = 4
add_op = add()
z = add_op(x, y)
grads_x, grads_y = add_op.backward(grads=1)
print('z         :', z)
print("x's grad is:", grads_x)
print("y's grad is:", grads_y)


#### 1.3.1.2 乘法算子

对于乘法 $z = x \times y$，反向计算的输入是 $\delta_z$，输出是 $\delta_x$ 和 $\delta_y$。根据链式法则：$\delta_x = \delta_z \times y$，$\delta_y = \delta_z \times x$。


In [ ]:
class multiply(Op):
    def __init__(self):
        super().__init__()

    def __call__(self, x, y):
        return self.forward(x, y)

    def forward(self, x, y):
        self.x = x
        self.y = y
        outputs = x * y
        return outputs

    def backward(self, grads):
        grads_x = grads * self.y
        grads_y = grads * self.x
        return grads_x, grads_y


#### 1.3.1.3 指数算子

对于指数运算 $z = \exp(x)$，反向计算的输入是 $\delta_z$，输出是 $\delta_x$。根据链式法则：$\delta_x = \delta_z \times \exp(x)$。


In [ ]:
import math


class exponential(Op):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        self.x = x
        outputs = math.exp(x)
        return outputs

    def backward(self, grads):
        grads = grads * math.exp(self.x)
        return grads


分别指定 $a, b, c, d$ 的值，通过实例化算子，调用加法、乘法和指数运算算子，计算得到 $g$。


In [ ]:
a, b, c, d = 2, 3, 2, 2

multiply_op = multiply()
add_op      = add()
exp_op      = exponential()

g = exp_op(add_op(multiply_op(a, b), multiply_op(c, d)))
print('g:', g)


**\\练一练\\**：手动执行上面算子的反向过程，验证梯度计算是否正确。提示：从最外层的 `exp_op` 开始，把上游梯度（初始为 1）一层层往下传，并和每个算子的局部梯度相乘。


### 1.3.2 自动微分机制

目前大部分深度学习平台都支持**自动微分**（Automatic Differentiation），即根据 `forward()` 函数来自动构建 `backward()` 函数。

> **自动微分的原理**：将所有的数值计算都分解为基本的原子操作，并构建**计算图**（computational graph）。计算图上的每个节点都是一个原子操作，保留前向和反向的计算结果，于是可以方便地通过链式法则来计算梯度。自动微分的详细介绍可参考《神经网络与深度学习》第 4.5 节。

PyTorch 的自动微分通过 trace 的方式记录各种算子和张量的前向计算，并自动创建相应的反向函数和反向张量来实现反向梯度的计算。

下面用一个比较简单的例子来了解整个过程。定义两个张量 `a` 和 `b`，并用 `requires_grad` 属性设置是否传递梯度：

- 当 `a` 的 `requires_grad=True` 时，会自动为 `a` 创建一个反向张量；
- 当 `b` 的 `requires_grad=False`（默认）时，不会为 `b` 创建反向张量。

> 与 PaddlePaddle 中的 `stop_gradient` 含义相反：`requires_grad=True` 等价于 `stop_gradient=False`，意思是"参与梯度计算"。


In [ ]:
# 定义张量 a，requires_grad=True 代表参与梯度计算
a = torch.tensor(2.0, requires_grad=True)
# 定义张量 b，requires_grad=False 代表不参与梯度计算
b = torch.tensor(5.0, requires_grad=False)

c = a * b
# 自动计算反向梯度
c.backward()

print("Tensor a's grad is:", a.grad)
print("Tensor b's grad is:", b.grad)   # 没有梯度，输出 None
print("Tensor c's grad is:", c.grad)   # 中间张量默认不保留 grad


下面解释一下上面代码的执行逻辑。

**前向执行**：在 `c.backward()` 被执行之前，PyTorch 已经为每个参与梯度计算的张量和算子构建好了一张**动态计算图**。

1. 创建张量 `a` 时，由于 `requires_grad=True`，会为它在计算图中预留一个"梯度位"（`.grad`），并把它登记为**叶子节点**。
2. 创建张量 `b` 时，`requires_grad=False`，不会为它在图里登记梯度位。
3. 执行乘法 `c = a * b` 时，乘法 `Mul` 是一个前向算子，PyTorch 会自动为它构建反向算子 `MulBackward`，并把 `c.grad_fn` 指向这个反向算子。
4. 反向算子里记录了它需要的上下文（这里是 `b` 的值），以便在 `c.backward()` 被调用时把上游梯度乘以 `b`，回传给 `a`。

由于此时还没有进行反向计算，因此 `a.grad` 等仍然是 `None`。

**反向执行**：调用 `c.backward()` 后，PyTorch 沿计算图自底向上执行反向算子，通过链式法则计算每个叶子节点的梯度，并把结果写到对应张量的 `.grad` 属性上。


**梯度累加与清零**：PyTorch 的梯度默认**累加**到 `.grad` 上，所以训练循环里每次 `backward()` 之前都要先把梯度清零。


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# 第一次 backward
y = (x ** 2).sum()
y.backward()
print('1st backward, dy/dx =', x.grad)   # [2., 4., 6.]

# 不清零，再做一次 —— 梯度会累加
y2 = (x ** 2).sum()
y2.backward()
print('2nd backward, dy/dx =', x.grad)   # [4., 8., 12.] —— 累加了

# 训练循环里要先清零再 backward
x.grad.zero_()
(x ** 2).sum().backward()
print('after zero_() , dy/dx =', x.grad)  # 回到 [2., 4., 6.]


**关闭梯度**：推理（前向预测）阶段不需要梯度，可以用 `torch.no_grad()` 包起来；想从一个张量切出"不参与梯度"的副本时，用 `tensor.detach()`。


In [ ]:
# 推理阶段关掉 autograd，节省内存
with torch.no_grad():
    pred = x * 2
print('pred.requires_grad:', pred.requires_grad)

# detach：切断计算图，但仍共享存储
z = (x * 3).detach()
print('z.requires_grad   :', z.requires_grad)


### 1.3.3 预定义的算子

从零开始构建各种复杂的算子和模型是一个很烦琐的过程，在开发时也容易出现冗余代码。因此 PyTorch 提供了大量预先实现好的基础算子（如 `+`、`-`、`*`、`/`、`exp` 等）和中间算子（如线性变换、Logistic 函数、卷积等），帮助我们便捷地实现复杂模型。

在深度学习中，大多数模型都是以各种神经网络为主，由一系列**层**（Layer）组成。PyTorch 提供了 `torch.nn.Module` 类来方便快速地实现自己的层和模型。模型和层都可以基于 `torch.nn.Module` 扩充实现，模型只是一种特殊的层。

当我们实现的算子继承 `torch.nn.Module` 类时，就不用再定义 `backward` 函数。PyTorch 的**自动微分机制**会自动完成反向传播过程，让我们只关注模型构建的前向过程，不必再进行烦琐的梯度求导。


### 1.3.4 本书中实现的算子

为了更深入地理解深度学习的模型和算法，本书中我们也手动实现自己的算子库 `nndl`，并基于自己的算子库来构建机器学习模型。本书中的自定义算子分为两类：

- 一类继承上节中定义的 `Op` 类。这些算子是为了更好地展示模型实现细节，需要自己动手计算并实现其反向函数。
- 另一类继承 `torch.nn.Module` 类，更方便搭建复杂算子，并与 PyTorch 预定义算子混合使用。

后续章节会陆续实现这些算子，包括 `Linear`、`Logistic`、`BinaryCrossEntropy`、`Conv2D`、`Pool2D`、`SRN`、`LSTM`、`MultiHeadSelfAttention`、`TransformerBlock` 等。完整列表见配套教材的对应章节。


## 1.4 数据集与 `Dataset` 类

为了更好地实践，本书在模型解读部分主要使用简单任务和数据集，在案例实践部分主要使用公开的实际案例数据集。本书中使用的数据集包括：

- **线性回归数据集 ToyLinear150**：用于简单的线性回归任务。
- **非线性回归数据集 ToySin25**：用于多项式回归任务。
- **波士顿房价预测数据集**：506 条样本数据，每条样本包含 12 种可能影响房价的因素和该类房屋价格的中位数。
- **二分类数据集 Moon1000**：从两个带噪声的弯月形状数据分布中采样得到。
- **三分类数据集 Multi1000**：来自三个不同的簇，每个簇对应一个类别。
- **鸢尾花数据集**：包含 3 种鸢尾花类别，每种 50 个样本，共 150 个样本。
- **MNIST 数据集**：手写体数字识别数据集，训练集 60,000 条、测试集 10,000 条。
- **CIFAR-10 数据集**：包含 10 种不同类别，共 60,000 张 $32 \times 32$ 图像。
- **IMDB 电影评论数据集**：二分类数据集，训练/测试各 25,000 条。
- **数字求和数据集 DigitSum**：测试模型对序列数据记忆能力的简单任务。
- **LCQMC 通用领域问题匹配数据集**：来自百度知道的中文问题匹配数据集。

为了更好地支持使用随机梯度下降法进行参数学习，PyTorch 提供了 `torch.utils.data.Dataset` 和 `DataLoader` 两个抽象。


### 1.4.1 自定义 `Dataset`

自定义数据集只需要继承 `Dataset` 类并实现两个方法：

- `__len__(self)`：返回数据集大小。
- `__getitem__(self, i)`：返回第 `i` 个样本（通常是 `(x, y)` 二元组）。

剩下的批次切分、随机打乱、并行加载交给 `DataLoader` 即可。


In [ ]:
from torch.utils.data import Dataset, DataLoader


class XOR(Dataset):
    """一个 4 条样本的玩具数据集，用来演示 Dataset 接口。"""

    def __init__(self):
        self.X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
        self.y = torch.tensor([0, 1, 1, 0])

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


ds = XOR()
print('数据集大小:', len(ds))
print('第 0 个样本:', ds[0])


### 1.4.2 用 `DataLoader` 做批量化

`DataLoader` 把 `Dataset` 包装成可迭代对象，常用参数：

- `batch_size`：每个批次的样本数。
- `shuffle`：是否在每个 epoch 打乱顺序。一般训练集设为 True，测试集为 False。
- `num_workers`：使用多少个子进程并行加载（Windows 下设 0 比较稳妥）。
- `drop_last`：批数不整除时是否丢掉最后一批。


In [ ]:
loader = DataLoader(ds, batch_size=2, shuffle=True)

for batch_idx, (x, y) in enumerate(loader):
    print(f'batch {batch_idx}: x={x.tolist()} y={y.tolist()}')


## 1.5 一个最小的训练循环：`nn.Module` 串起来

我们把前面学到的工具组合起来，训练一个最小的回归模型。整个流程是：

1. 用 `nn.Module` 定义网络结构；
2. 准备数据（这里直接生成合成数据）；
3. 选定损失函数和优化器；
4. 写训练循环：前向 → 求 loss → 反向 → 更新参数。


In [ ]:
import torch.nn as nn
import torch.optim as optim


class TinyMLP(nn.Module):
    """一个两层的小型多层感知机，用来拟合 y = x1 + x2 + 噪声。"""

    def __init__(self, in_dim=2, hid=16, out=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hid),
            nn.ReLU(),
            nn.Linear(hid, out),
        )

    def forward(self, x):
        return self.net(x)


torch.manual_seed(0)

# 合成回归数据 y = sum(x) + 噪声
X = torch.randn(256, 2)
y = X.sum(dim=1, keepdim=True) + 0.1 * torch.randn(256, 1)

model   = TinyMLP()
opt     = optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()

for epoch in range(200):
    pred = model(X)                # 前向
    loss = loss_fn(pred, y)        # 求 loss
    opt.zero_grad()                # 清零梯度
    loss.backward()                # 反向
    opt.step()                     # 更新参数

    if epoch % 50 == 0:
        print(f'epoch {epoch:3d}  loss {loss.item():.4f}')


## 1.6 本书中实现的 `Runner` 类

在一个任务上应用机器学习方法的流程一般包括数据集构建、模型构建、损失函数定义、优化器定义、评价指标定义、模型训练、模型评价和模型预测等环节。为了将上述环节规范化，本书将机器学习模型的基本要素封装成一个 `Runner` 类，以便更方便地进行机器学习实践。除了上述要素外，`Runner` 类还包括模型保存、模型加载等功能。

本书会循序渐进地实现 3 个版本的 `Runner`：

| 版本 | 关键能力 | 引入章节 |
|---|---|---|
| `RunnerV1` | 直接调最小二乘法解析解；只做保存 | 本节末尾 |
| `RunnerV2` | 加入梯度下降法训练；记录 train/dev 损失;保存最优模型 | 第 3 章 |
| `RunnerV3` | 用 `DataLoader` 做小批量训练；`state_dict` 保存 | 第 4 章 |

`RunnerV3` 已经够用于绝大多数任务，后续章节都基于它做轻量扩展。下面我们先实现 `RunnerV1`，V2 / V3 留到第 3、4 章引入。

### 1.6.1 `RunnerV1` 的实现

`RunnerV1` 的职责非常简单：调用一个**求解器**（optimizer）在 $(X, y)$ 上一步算出解析解，再把模型参数保存到磁盘。

- `model`：自定义算子（持有 `params` 字典），实现 `__call__(X)` 做前向；
- `optimizer`：一个函数 `optimizer(model, X, y, **kwargs)`，在 `model.params` 上原地写入闭式解（下一章会看到一个具体例子 `optimizer_lsm`）；
- `fit(X, y)`：调用一次 `optimizer`，完成训练；
- `predict(X)`：前向预测；
- `evaluate(X, y, metric_fn)`：用给定的指标函数评价；
- `save(path)` / `load(path)`：把 `params` 字典存/取磁盘。

In [ ]:
from pathlib import Path

class RunnerV1:
    def __init__(self, model, optimizer):
        self.model = model
        self.optimizer = optimizer

    def fit(self, X, y, **kwargs):
        # 求解器在 model.params 上原地写入闭式解
        self.optimizer(self.model, X, y, **kwargs)

    def predict(self, X):
        return self.model(X)

    def evaluate(self, X, y, metric_fn):
        return metric_fn(y, self.model(X)).item()

    def save(self, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        torch.save(self.model.params, path)

    def load(self, path):
        self.model.params = torch.load(path, weights_only=True)

> **提示**：本书把这三个版本的 `Runner` 也打包到了 `nndl/runner.py` 里，后续章节可以直接 `from nndl.runner import RunnerV1` 复用；这里在 notebook 内 inline 一份实现，是为了让你能在一处看清全貌。

### 1.6.2 演示：用 `RunnerV1` 拟合一组合成线性数据

构造 $y = 2x + 1 + \epsilon$ 的合成数据，定义一个最小的 `Linear` 算子（用 `params` 字典持有权重和偏置），再写一个 `optimizer_lsm` 函数实现一元线性回归的闭式解：

$$ w^* = \frac{\sum_i (x_i - \bar x)(y_i - \bar y)}{\sum_i (x_i - \bar x)^2},\qquad b^* = \bar y - \bar x\, w^*. $$

把它们交给 `RunnerV1` 一行求解。

In [ ]:
torch.manual_seed(0)

# 合成数据 y = 2x + 1 + 噪声
lr_N = 100
lr_X = torch.linspace(-3, 3, lr_N).unsqueeze(1)
lr_y = 2 * lr_X + 1 + 0.3 * torch.randn(lr_N, 1)

# 最小的 Linear 算子：params 字典里持有 w, b
class Linear:
    def __init__(self, input_size):
        self.params = {'w': torch.zeros(input_size, 1), 'b': torch.zeros(1)}
    def __call__(self, X):
        return X @ self.params['w'] + self.params['b']

# 闭式解求解器（最小二乘法）
def optimizer_lsm(model, X, y):
    x_bar = X.mean(dim=0, keepdim=True)
    y_bar = y.mean()
    A = (X - x_bar).T @ (X - x_bar)
    rhs = (X - x_bar).T @ (y - y_bar)
    w = torch.linalg.solve(A, rhs)
    b = y_bar - x_bar @ w
    model.params['w'] = w
    model.params['b'] = b.squeeze(0)

# 训练
lr_model = Linear(input_size=1)
lr_runner = RunnerV1(lr_model, optimizer_lsm)
lr_runner.fit(lr_X, lr_y)
print(f"w = {lr_model.params['w'].item():.4f}   b = {lr_model.params['b'].item():.4f}")

# 评估
def mse(y_true, y_pred):
    return ((y_true - y_pred) ** 2).mean()

print(f'train MSE: {lr_runner.evaluate(lr_X, lr_y, mse):.4f}')

# 保存 / 重新加载
import tempfile
with tempfile.TemporaryDirectory() as td:
    save_path = Path(td) / 'linear.pt'
    lr_runner.save(save_path)

    lr_model2 = Linear(input_size=1)
    lr_runner2 = RunnerV1(lr_model2, optimizer_lsm)
    lr_runner2.load(save_path)
    print(f'reloaded train MSE: {lr_runner2.evaluate(lr_X, lr_y, mse):.4f}')

拟合结果 $(w, b) \approx (2, 1)$ 与真值一致；保存后重新加载的模型评估出同样的 MSE，说明 `save` / `load` 都没问题。

第 2 章会用同样的 `RunnerV1` 跑加州房价的多元线性回归（特征维度 $D=8$，`optimizer_lsm` 推广到一般的正规方程）。

## 1.7 模型的保存与加载

训练好的模型需要保存下来，以便后续继续训练或部署到生产环境。PyTorch 推荐保存模型的 `state_dict`，而不是整个 `model` 对象——前者只保存参数张量，跨结构改动更稳定；后者依赖具体的类定义和路径，反序列化时容易出问题。

常用 API：

- `torch.save(obj, path)`：把任意可序列化对象（张量、字典、模型 `state_dict` 等）保存到文件。
- `torch.load(path, weights_only=True)`：从文件读出对象。`weights_only=True` 是 PyTorch ≥ 2.4 推荐的安全选项，禁止反序列化任意 Python 对象。
- `Module.state_dict()`：返回一个 OrderedDict，键是参数名（如 `net.0.weight`），值是对应的张量。
- `Module.load_state_dict(d)`：用字典 `d` 里的参数覆盖当前模型的参数。


In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as td:
    save_path = Path(td) / 'tinymlp.pt'

    # 保存：把 state_dict 写到文件
    torch.save(model.state_dict(), save_path)

    # 加载：先新建一个同结构的模型，再 load_state_dict
    model2 = TinyMLP()
    model2.load_state_dict(torch.load(save_path, weights_only=True))
    model2.eval()           # 切换到推理模式（关掉 Dropout / BN 的训练行为）

    # 验证：两份模型在同样输入下输出应当一致
    with torch.no_grad():
        a = model(X[:5])
        b = model2(X[:5])
    print('max abs diff:', (a - b).abs().max().item())


## 1.8 小结

本章介绍了我们在后面实践中需要的一些基础知识，包括：

- 张量的概念、创建方式、形状操作、运算与广播；
- 算子的定义、手动实现前向/反向过程，以及 PyTorch 的自动微分机制；
- `nn.Module` 与最小训练循环；
- `Dataset` / `DataLoader` 数据加载抽象；
- 本书 `Runner` 类的三版本规划与 `RunnerV1` 的实现；
- 模型的保存与加载。

如需查阅张量、算子或其他 PyTorch 知识，可参阅 [PyTorch 官方文档](https://pytorch.org/docs/stable/index.html)。后续章节会逐步使用本章介绍的工具构建越来越复杂的模型。